# Synthetic E-Commerce Support Data Generator

Generates relational test data for PostgreSQL tables defined in `docs/sql_db_query.md`:
1. **`customers`** (50 records)
2. **`orders`** (80 records)
3. **`order_items`** (~150-200 line items)
4. **`conversation_logs`** (Generated dynamically by AI Agent sessions)

In [ ]:
import json
import random
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd
from pathlib import Path

# Setup Faker & Seed for reproducibility
fake = Faker("en_IN")
Faker.seed(42)
random.seed(42)

In [ ]:
# Sample Products Catalog & Couriers
PRODUCTS = [
    ("Wireless Bluetooth Earbuds", 1499.00),
    ("USB-C Fast Charging Cable", 399.00),
    ("Smart Fitness Watch Series 5", 2999.00),
    ("Ergonomic Wireless Mouse", 799.00),
    ("Mechanical Gaming Keyboard", 2499.00),
    ("65W GaN Fast Charger", 1299.00),
    ("Waterproof Laptop Backpack", 1899.00),
    ("Noise Cancelling Headphones", 4599.00),
    ("Stainless Steel Water Bottle", 599.00),
    ("Desk LED Lamp", 899.00)
]

COURIERS = ["Delhivery", "BlueDart", "Shadowfax", "XpressBees"]

ORDER_STATUSES = ["Placed", "Shipped", "Delivered", "Cancelled", "Returned"]

STATUS_WEIGHTS = [20, 35, 30, 10, 5]

In [ ]:
# 1. Generate Customers (50 records)

NUM_CUSTOMERS = 50
customers = []

for cid in range(1, NUM_CUSTOMERS + 1):
    customers.append({
        "customer_id": cid,
        "name": fake.name(),
        "email": fake.unique.email(),
        "phone": f"+91{random.randint(6000000000, 9999999999)}",
        "date_of_birth": fake.date_of_birth(minimum_age=18, maximum_age=65),
        "created_at": datetime.now() - timedelta(days=random.randint(30, 180))
    })

df_customers = pd.DataFrame(customers)
print(f"[OK] Customers generated: {len(df_customers)} rows")

In [ ]:
# 2. Generate Orders & Order Items (80 orders)
NUM_ORDERS = 80
orders = []
order_items = []
item_id_counter = 1

for oid in range(1, NUM_ORDERS + 1):
    cust_id = random.randint(1, NUM_CUSTOMERS)
    status = random.choices(ORDER_STATUSES, weights=STATUS_WEIGHTS, k=1)[0]
    
    order_date = datetime.now() - timedelta(days=random.randint(1, 30))
    
    # Courier & Tracking logic
    courier = random.choice(COURIERS) if status in ["Shipped", "Delivered", "Returned"] else None
    tracking = f"{courier[:3].upper()}{random.randint(100000, 999999)}" if courier else None
    
    # Delivery date populated for active/delivered orders (None if Cancelled)
    delivery_date = (
        order_date + timedelta(days=random.randint(2, 5))
        if status != "Cancelled"
        else None
    )

    # 1 to 3 items per order
    selected_products = random.sample(PRODUCTS, k=random.randint(1, 3))
    total_amount = 0.0

    for prod_name, unit_price in selected_products:
        qty = random.randint(1, 3)
        total_amount += qty * unit_price
        item_status = status

        order_items.append({
            "order_item_id": item_id_counter,
            "order_id": oid,
            "product_name": prod_name,
            "quantity": qty,
            "unit_price": unit_price,
            "item_status": item_status
        })
        item_id_counter += 1

    orders.append({
        "order_id": oid,
        "customer_id": cust_id,
        "order_date": order_date,
        "delivery_date": delivery_date,
        "status": status,
        "total_amount": round(total_amount, 2),
        "tracking_number": tracking,
        "courier_name": courier,
        "created_at": order_date,
        "updated_at": order_date
    })

df_orders = pd.DataFrame(orders)
df_order_items = pd.DataFrame(order_items)

print(f"[OK] Orders generated: {len(df_orders)} rows")
print(f"[OK] Order Items generated: {len(df_order_items)} rows")

In [ ]:
df_customers.head()

In [ ]:
# View orders summary
df_orders.sort_values(by='customer_id').head(5)

In [ ]:
df_order_items

In [ ]:
df_logs.head()

In [ ]:
# 4. Save 3 Core DataFrames to CSV in datasets/ directory

from pathlib import  Path
output_dir = Path(r'D:\Customer-Support-AI-Agent\datasets')
output_dir.mkdir(parents=True, exist_ok=True)

df_customers.to_csv(output_dir / "customers.csv", index=False)
df_orders.to_csv(output_dir / "orders.csv", index=False)
df_order_items.to_csv(output_dir / "order_items.csv", index=False)

print(f"[OK] Successfully saved customers.csv, orders.csv, and order_items.csv to: {output_dir.resolve()}")

[OK] Successfully saved customers.csv, orders.csv, and order_items.csv to: D:\Customer-Support-AI-Agent\datasets
